In [1]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — Install / imports
# ══════════════════════════════════════════════════════════════════
import os
import shutil
import itertools
import time
from datetime import datetime

import pandas as pd
from kaggle_secrets import UserSecretsClient

# Retrieve HF token from Kaggle Secrets (never hard-code it)
_secrets   = UserSecretsClient()
HF_TOKEN   = _secrets.get_secret("HF_TOKEN")

DATASET_SLUG   = "lorentzyeung/price-paid-data-202304"
TEMP_DIR       = "./temp_kaggle_data/"
SUBDATA_DIR    = "./database/subdata/"
HF_SPACE_ID    = "Entz/uk-property"
HF_REPO_PREFIX = "database/subdata"

os.makedirs(SUBDATA_DIR, exist_ok=True)

In [2]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — Check if an update is needed
# ══════════════════════════════════════════════════════════════════

def _parse_kaggle_version(txt: str) -> str | None:
    """
    Convert Kaggle last_update.txt content like 'January 2026 data'
    to a YYYY-MM string like '2026-01' for comparison with meta.txt.
    Returns None if parsing fails.
    """
    try:
        month_part = txt.replace(" data", "").strip()   # "January 2026"
        dt = datetime.strptime(month_part, "%B %Y")
        return dt.strftime("%Y-%m")
    except ValueError:
        return None


def get_kaggle_version() -> str | None:
    """Download dataset and read last_update.txt."""
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)
    os.makedirs(TEMP_DIR)
    os.system(f"kaggle datasets download -d {DATASET_SLUG} -p {TEMP_DIR} --unzip")
    path = os.path.join(TEMP_DIR, "last_update.txt")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return _parse_kaggle_version(f.read().strip())


def get_hf_version() -> str | None:
    """Read meta.txt from the HF Space. Returns None if not yet uploaded."""
    from huggingface_hub import hf_hub_download
    from huggingface_hub.utils import EntryNotFoundError
    try:
        local = hf_hub_download(
            repo_id   = HF_SPACE_ID,
            filename  = f"{HF_REPO_PREFIX}/meta.txt",
            repo_type = "space",
            token     = HF_TOKEN,
        )
        with open(local) as f:
            for line in f:
                if line.startswith("last_updated="):
                    return line.strip().split("=", 1)[1]
    except EntryNotFoundError:
        pass
    return None


kaggle_version = get_kaggle_version()
hf_version     = get_hf_version()

print(f"Kaggle dataset version : {kaggle_version}")
print(f"HF Space version       : {hf_version}")

NEED_UPDATE = (kaggle_version is not None) and (kaggle_version != hf_version)
print(f"Update needed          : {NEED_UPDATE}")

Dataset URL: https://www.kaggle.com/datasets/lorentzyeung/price-paid-data-202304
License(s): other


100%|██████████| 900M/900M [00:05<00:00, 163MB/s]


meta.txt:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

Kaggle dataset version : 2026-01
HF Space version       : 2026-01
Update needed          : False


In [3]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — Generate parquets (only runs if NEED_UPDATE)
# ══════════════════════════════════════════════════════════════════

# ── Identical logic to pipeline/generate_subdata.py ───────────────
COLS = [
    "price", "Date_of_Transfer", "postcode", "Property_Type", "Old_New",
    "Duration", "PAON", "SAON", "Street", "Locality", "Town_City",
    "District", "County", "PPDCategory_Type", "Record_Status",
]
PRICE_BANDS  = [0, 250_000, 500_000, 1_000_000, float("inf")]
BAND_LABELS  = ["Under 250k", "250k-500k", "500k-1M", "Over 1M"]
STAT_DIMS    = ["Property_Type", "Old_New", "Tenure"]
ALL_DIM_COLS = ["District", "postcodeA", "Property_Type", "Old_New", "Tenure"]
YEAR_FROM    = 2010


def load(csv_path: str) -> pd.DataFrame:
    print(f"Reading {csv_path} ...")
    raw = pd.read_csv(csv_path, header=None, names=COLS, low_memory=False)
    print(f"  Loaded {len(raw):,} rows")
    df = raw[raw["County"] == "GREATER LONDON"][
        ["price", "Date_of_Transfer", "postcode", "Property_Type",
         "Old_New", "Duration", "District"]
    ].copy()
    df["price"]            = pd.to_numeric(df["price"], errors="coerce")
    df["Date_of_Transfer"] = pd.to_datetime(df["Date_of_Transfer"], errors="coerce")
    df.dropna(subset=["price", "District", "Date_of_Transfer"], inplace=True)
    df["year"]      = df["Date_of_Transfer"].dt.year
    df["month"]     = df["Date_of_Transfer"].dt.month
    df["y_m"]       = df["Date_of_Transfer"].dt.to_period("M").dt.to_timestamp()
    df["postcodeA"] = df["postcode"].str.strip().str.split().str[0]
    df["Tenure"]    = df["Duration"].where(df["Duration"].isin(["F", "L"]))
    df = df[df["year"] >= YEAR_FROM].copy()
    print(f"  Greater London rows (2010+): {len(df):,}")
    return df


def _agg_price(df, group_cols):
    agg = (
        df.groupby(group_cols, observed=True)
        .agg(count=("price","count"), median_price=("price","median"),
             mean_price=("price","mean"), min_price=("price","min"), max_price=("price","max"))
        .reset_index()
    )
    for col in ALL_DIM_COLS:
        if col not in agg.columns: agg[col] = None
    return agg


def _stat_combos():
    combos = []
    for r in range(len(STAT_DIMS) + 1):
        combos.extend(itertools.combinations(STAT_DIMS, r))
    return combos


def make_txn_monthly(df):
    results = []
    for combo in _stat_combos():
        for loc in [[], ["District"], ["District", "postcodeA"]]:
            results.append(_agg_price(df, loc + list(combo) + ["y_m"]))
    combined = pd.concat(results, ignore_index=True)
    combined["year"]  = combined["y_m"].dt.year
    combined["month"] = combined["y_m"].dt.month
    front = ["District","postcodeA","Property_Type","Old_New","Tenure","y_m","year","month"]
    return combined[front + ["count","median_price","mean_price","min_price","max_price"]]


def make_txn_annual(df):
    results = []
    for combo in _stat_combos():
        for loc in [[], ["District"], ["District", "postcodeA"]]:
            results.append(_agg_price(df, loc + list(combo) + ["year"]))
    combined = pd.concat(results, ignore_index=True)
    front = ["District","postcodeA","Property_Type","Old_New","Tenure","year"]
    return combined[front + ["count","median_price","mean_price","min_price","max_price"]]


def make_affordability_annual(df):
    tmp = df.copy()
    tmp["band"] = pd.cut(tmp["price"], bins=PRICE_BANDS, labels=BAND_LABELS, right=False)
    def _agg_bands(sub, group_cols):
        counts = sub.groupby(group_cols+["band"], observed=True).size().reset_index(name="count")
        totals = counts.groupby(group_cols)["count"].transform("sum")
        counts["pct_of_total"] = (counts["count"] / totals * 100).round(2)
        for col in ALL_DIM_COLS:
            if col not in counts.columns: counts[col] = None
        return counts
    results = []
    for combo in _stat_combos():
        for loc in [[], ["District"], ["District", "postcodeA"]]:
            results.append(_agg_bands(tmp, loc + list(combo) + ["year"]))
    combined = pd.concat(results, ignore_index=True)
    front = ["District","postcodeA","Property_Type","Old_New","Tenure","year","band"]
    return combined[front + ["count","pct_of_total"]].sort_values(front)


if NEED_UPDATE:
    csv_path = os.path.join(TEMP_DIR, "pp-complete.csv")
    df = load(csv_path)

    for name, fn in [
        ("txn_monthly",          make_txn_monthly),
        ("txn_annual",           make_txn_annual),
        ("affordability_annual", make_affordability_annual),
    ]:
        t0 = time.time()
        print(f"Building {name} ...")
        frame = fn(df)
        out   = os.path.join(SUBDATA_DIR, f"{name}.parquet")
        frame.to_parquet(out, index=False)
        mb = os.path.getsize(out) / 1_048_576
        print(f"  {frame.shape[0]:,} rows | {mb:.1f} MB | {time.time()-t0:.0f}s")

    # Write meta.txt
    max_year  = int(df["year"].max())
    max_month = int(df[df["year"] == max_year]["month"].max())
    with open(os.path.join(SUBDATA_DIR, "meta.txt"), "w") as f:
        f.write(f"last_updated={max_year}-{max_month:02d}\n")
        f.write(f"min_year={YEAR_FROM}\n")
        f.write(f"max_year={max_year}\n")
        f.write(f"total_rows={len(df)}\n")
    print("meta.txt written.")
else:
    print("Skipping generation — HF Space is already up to date.")


Skipping generation — HF Space is already up to date.


In [4]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — Upload to HF Space (only runs if NEED_UPDATE)
# ══════════════════════════════════════════════════════════════════

if NEED_UPDATE:
    from huggingface_hub import HfApi
    api = HfApi()

    for filename in ["txn_monthly.parquet", "txn_annual.parquet",
                     "affordability_annual.parquet", "meta.txt"]:
        local = os.path.join(SUBDATA_DIR, filename)
        repo_path = f"{HF_REPO_PREFIX}/{filename}"
        mb = os.path.getsize(local) / 1_048_576
        print(f"Uploading {filename} ({mb:.1f} MB) → {repo_path} ...")
        api.upload_file(
            path_or_fileobj = local,
            path_in_repo    = repo_path,
            repo_id         = HF_SPACE_ID,
            repo_type       = "space",
            token           = HF_TOKEN,
        )
        print(f"  ✓ done")

    print("\nAll files uploaded. HF backend will reload parquets on next query.")
else:
    print("Nothing to upload.")


Nothing to upload.
